In [ ]:
!git clone https://github.com/antonawinkler/slm-pricer.git

%cd slm-pricer
!uv pip install .
%cd ..

In [ ]:
import os
from datetime import datetime

import torch
import wandb
from datasets import Dataset
from google.colab import userdata
from peft import LoraConfig, TaskType, get_peft_model
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    models,
)
from transformers import BitsAndBytesConfig

from slm_pricer.data import (
    load_data_from_hf,
)

In [ ]:
# ============================================
# HYPERPARAMETERS - Configure all settings here
# ============================================

# Model Configuration
MODEL_NAME = "meta-llama/Llama-3.2-3B"
MAX_SEQ_LENGTH = 128
POOLING_MODE = "lasttoken"

# Quantization
USE_4BIT_QUANT = True
USE_FLASH_ATTENTION = True

# SimCSE Dropout (critical - Llama defaults to 0.0)
SIMCSE_DROPOUT = 0.1

# LoRA Configuration
LORA_R = 16
LORA_ALPHA = 32  # 2 * LORA_R
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    # "gate_proj",
    # "up_proj",
    # "down_proj",
]

# Training Hyperparameters
EPOCHS = 2
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.001
MAX_GRAD_NORM = 0.3

# Loss Function
USE_CACHED_LOSS = False  # True = more negatives but 2x slower
MINI_BATCH_SIZE = 32  # Only used if USE_CACHED_LOSS=True

# Logging & Checkpointing
LOGGING_STEPS = 100
EVAL_FREQUENCY = 5  # Evaluate N times per epoch
SAVE_FREQUENCY = 1  # Save N times per epoch

# Data
DATASET_NAME = "ed-donner/items_full"
DATA_PERCENT = 100

# Output & Hub
OUTPUT_DIR = "llama-3.2-3b-lora-output"
MODEL_REPO_ID = "antonawinkler/slm-pricer-llama-3.2-3b"
RUN_NAME_BASE = "llama-3.2-3b-simcse"
RUN_NAME = f"{RUN_NAME_BASE}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
WANDB_PROJECT = "slm-pricer-llama-3b"

print("=" * 50)
print("CONFIGURATION LOADED")
print("=" * 50)
print(f"Model: {MODEL_NAME}")
print(f"Run name: {RUN_NAME}")
print(f"SimCSE dropout: {SIMCSE_DROPOUT}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Batch size: {BATCH_SIZE}, Eval: {EVAL_BATCH_SIZE}")
print(f"Epochs: {EPOCHS}, LR: {LEARNING_RATE}")
print(f"Cached loss: {USE_CACHED_LOSS}")
print("=" * 50)

In [ ]:
wandb_api_key = userdata.get("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()
os.environ["WANDB_LOG_MODEL"] = "checkpoint"
os.environ["WANDB_WATCH"] = "gradients"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT_QUANT,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

word_embedding_model = models.Transformer(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    model_args={
        "quantization_config": bnb_config,
        "torch_dtype": torch.bfloat16,
        "trust_remote_code": True,
        "attn_implementation": "flash_attention_2" if USE_FLASH_ATTENTION else None,
    },
    config_args={
        "attention_dropout": SIMCSE_DROPOUT,
    },
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode=POOLING_MODE,
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

In [ ]:
df_train = load_data_from_hf(
    split="train", percent=DATA_PERCENT, dataset_name=DATASET_NAME
)
df_val = load_data_from_hf(
    split="validation", percent=DATA_PERCENT, dataset_name=DATASET_NAME
)

# SimCSE: Use identical text for sentence1 and sentence2
df_train["sentence1"] = df_train["summary"]
df_train["sentence2"] = df_train["summary"]

df_val["sentence1"] = df_val["summary"]
df_val["sentence2"] = df_val["summary"]

train_dataset = Dataset.from_pandas(df_train[["sentence1", "sentence2"]])
eval_dataset = Dataset.from_pandas(df_val[["sentence1", "sentence2"]])

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Validation dataset: {len(eval_dataset):,} samples")

In [ ]:
if model[0].tokenizer.pad_token is None:
    model[0].tokenizer.pad_token = model[0].tokenizer.eos_token

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
)

model[0].auto_model = get_peft_model(model[0].auto_model, peft_config)

In [ ]:
wandb.init(project=WANDB_PROJECT, name=RUN_NAME)

In [ ]:
# Calculate steps per epoch
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
EVAL_STEPS = steps_per_epoch // EVAL_FREQUENCY
SAVE_STEPS = steps_per_epoch // SAVE_FREQUENCY

print(f"Dataset size: {len(train_dataset):,} samples")
print(f"Steps per epoch: {steps_per_epoch:,}")
print(f"Eval steps: {EVAL_STEPS:,} ({EVAL_FREQUENCY} times per epoch)")
print(f"Save steps: {SAVE_STEPS:,} ({SAVE_FREQUENCY} times per epoch)")

# Choose loss function
if USE_CACHED_LOSS:
    train_loss = losses.CachedMultipleNegativesRankingLoss(
        model=model, mini_batch_size=MINI_BATCH_SIZE
    )
    print(
        f"Using CachedMultipleNegativesRankingLoss: {BATCH_SIZE - 1} negatives (2x slower)"
    )
else:
    train_loss = losses.MultipleNegativesRankingLoss(model=model)
    print(f"Using MultipleNegativesRankingLoss: {BATCH_SIZE - 1} negatives")

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    optim="adamw_torch",
    fp16=False,
    bf16=True,
    max_grad_norm=MAX_GRAD_NORM,
    group_by_length=True,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    report_to="wandb",
    run_name=RUN_NAME,
    push_to_hub=True,
    hub_model_id=MODEL_REPO_ID,
    hub_private_repo=True,
    hub_strategy="every_save",
)

trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=train_loss,
    args=args,
)

trainer.train()

wandb.finish()